# PRAKTIKUM DEEP LEARNING — TUGAS UTS
## Eksperimen Optimasi Bertingkat (Progressive Ablation Study): Arsitektur Convolutional Neural Network (CNN) pada Citra CT-Scan Kanker Paru-Paru (IQ-OTH/NCCD)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attaramadhani/TUGAS-UTS_DEEP-LEARNING-A/blob/main/Tugas_UTS_CNN_IQOTHNCCD.ipynb)

* **Dosen Pengampu:** Dr. Wahyudi Setiawan, S.Kom., M.Kom.
* **Dataset Sumber Terpercaya:** **Mendeley Data** — *The IQ-OTHNCCD Lung Cancer Dataset* (DOI: [10.17632/bhmdr45bh2.2](https://doi.org/10.17632/bhmdr45bh2.2))
* **Penulis Dataset:** Hamdalla Alyasriy & Muayed AL-Huseiny (Wasit University & IQ-OTH/NCCD Oncology Centers)
* **Framework:** TensorFlow 2.x / Keras & Python 3.10+

---

### 👥 Identitas Kelompok 6:
| No. | Nama Lengkap | NIM | Kelas | Program Studi |
| :---: | :--- | :---: | :---: | :---: |
| 1. | **Attala Alif Ramadhani Tri Hida** | `230441100144` (23-144) | Deep Learning (A) | Sistem Informasi |
| 2. | **Nafaul Hernanda Romadlona** | `240441100125` (24-125) | Deep Learning (A) | Sistem Informasi |
| 3. | **M.Rafly Kurniawan** | `240441100086` (24-086) | Deep Learning (A) | Sistem Informasi |
| 4. | **Naufal Husain** | `240441100038` (24-038) | Deep Learning (A) | Sistem Informasi |

---

### 🎯 Konsep Desain 4 Skenario Pengujian Bertingkat (Progressive Pipeline):
Sesuai arahan Dosen Pengampu, eksperimen ini mengevaluasi **1 arsitektur CNN yang sama** melalui alur optimasi berjenjang di mana hasil terbaik dari setiap tahap diwariskan ke tahap berikutnya:
1. **Skenario 1 (Optimasi Data Split):** Menguji rasio pembagian data (70:15:15 vs 80:10:10 vs 90:05:05) pada baseline $ightarrow$ **Pemenang Split lanjut ke Skenario 2**.
2. **Skenario 2 (Optimasi Data Augmentasi):** Mengambil split terbaik dari Skenario 1, lalu menguji secara langsung **Tanpa Augmentasi vs Dengan Augmentasi** $ightarrow$ **Pemenang Augmentasi lanjut ke Skenario 3**.
3. **Skenario 3 (Optimasi Optimizer):** Mengambil konfigurasi terbaik Skenario 1 & 2, lalu membandingkan **Adam vs RMSprop vs SGD Momentum** $ightarrow$ **Pemenang Optimizer lanjut ke Skenario 4**.
4. **Skenario 4 (Optimasi Regularisasi Dropout):** Mengambil konfigurasi terbaik Skenario 1, 2, dan 3, lalu menguji variasi nilai **Dropout (0.0 vs 0.3 vs 0.5)** $ightarrow$ **Menghasilkan FINAL CHAMPION MODEL**.


---
## 1. Setup Environment, Mount Google Drive & Inisialisasi Smart Cache
Menyiapkan modul TensorFlow, scikit-learn, PIL, matplotlib, mengonfigurasi sinkronisasi Google Drive otomatis, dan memuat memori eksperimen (`cache.pkl`) agar eksekusi instan.

In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT, MOUNT GOOGLE DRIVE, DIREKTORI, & SMART CACHE
# ==============================================================================
print("=" * 80)
print("[LANGKAH 1] Inisialisasi Environment, Mount Google Drive, & Smart Cache")
print("=" * 80)

# Instalasi dependensi di Google Colab jika belum terpasang
try:
    import kagglehub
except ImportError:
    get_ipython().system('pip install -q kagglehub')
    import kagglehub

try:
    import docx
except ImportError:
    get_ipython().system('pip install -q python-docx')
    import docx

import os
import sys
import time
import json
import shutil
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

# Tetapkan Seed Reproduksibilitas
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

BASE_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data', 'lung_cancer')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
FIGURES_DIR = os.path.join(OUTPUTS_DIR, 'figures')
MODELS_DIR = os.path.join(OUTPUTS_DIR, 'models')
LOGS_DIR = os.path.join(OUTPUTS_DIR, 'logs')

for d in [DATA_DIR, FIGURES_DIR, MODELS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------------------------
# KONFIGURASI MOUNT GOOGLE DRIVE & AUTO-DISCOVERY TARGET FOLDER
# ------------------------------------------------------------------------------
MOUNT_GDRIVE = True

# Jika Anda memiliki nama folder spesifik di Google Drive, isi di bawah ini (opsional).
# Contoh: "TUGAS UTS DEEP LEARNING" atau biarkan kosong ("") untuk auto-detect folder
NAMA_FOLDER_GDRIVE_KHUSUS = ""

gdrive_mounted = False
if MOUNT_GDRIVE:
    try:
        from google.colab import drive
        print("[INFO] Menghubungkan Google Drive di Google Colab...")
        drive.mount('/content/drive')
        gdrive_mounted = True
        print("✔ Google Drive berhasil di-mount.")
    except Exception as e:
        print(f"[INFO] Google Drive tidak aktif ({e}). Berjalan di lingkungan lokal.")

def get_all_target_dirs():
    targets = [OUTPUTS_DIR]
    if gdrive_mounted and os.path.exists('/content/drive/MyDrive'):
        # Target utama di Google Drive
        primary_gdrive = '/content/drive/MyDrive/TUGAS-UTS_DEEP-LEARNING-A_KELOMPOK_6/'
        if primary_gdrive not in targets:
            targets.append(primary_gdrive)
            
        # Jika ada nama folder khusus yang ditentukan user:
        if NAMA_FOLDER_GDRIVE_KHUSUS.strip():
            custom_dir = os.path.join('/content/drive/MyDrive', NAMA_FOLDER_GDRIVE_KHUSUS.strip())
            if custom_dir not in targets:
                targets.append(custom_dir)
                
        # Deteksi otomatis folder yang ada kata 'uts', 'deep learning', atau 'iqoth' di MyDrive
        try:
            for item in os.listdir('/content/drive/MyDrive'):
                item_lower = item.lower()
                if ('uts' in item_lower or 'deep learning' in item_lower or 'tugas-uts' in item_lower):
                    fpath = os.path.join('/content/drive/MyDrive', item)
                    if os.path.isdir(fpath) and fpath not in targets:
                        targets.append(fpath)
        except Exception:
            pass
            
    return targets

# Fungsi Simpan & Replace Otomatis ke Seluruh Target (Lokal + Google Drive)
def save_and_replace_figure(fig, filename, dpi=300):
    for d in get_all_target_dirs():
        dest_dir = os.path.join(d, 'figures') if not d.endswith('figures') else d
        os.makedirs(dest_dir, exist_ok=True)
        dest = os.path.join(dest_dir, filename)
        fig.savefig(dest, dpi=dpi, bbox_inches='tight')
        print(f"   ✔ [SIMPAN GDRIVE/LOKAL] Grafik: {filename} -> {dest}")

def save_and_replace_cache(data, filename='cache.pkl'):
    for d in get_all_target_dirs():
        os.makedirs(d, exist_ok=True)
        dest = os.path.join(d, filename)
        with open(dest, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"   ✔ [SIMPAN GDRIVE/LOKAL] Cache: {filename} -> {dest}")

def save_and_replace_docx(doc_obj, filename='Laporan_Lengkap_UTS_DeepLearning_CNN.docx'):
    for d in get_all_target_dirs():
        os.makedirs(d, exist_ok=True)
        dest = os.path.join(d, filename)
        doc_obj.save(dest)
        print(f"   ✔ [SIMPAN GDRIVE/LOKAL] Laporan Word: {filename} -> {dest} ({os.path.getsize(dest)/1024:.1f} KB)")

# ------------------------------------------------------------------------------
# SMART CACHE SYSTEM (MEMORI EKSPERIMEN INSTAN)
# ------------------------------------------------------------------------------
# Jika berjalan di Colab dan cache.pkl belum ada, unduh otomatis dari repositori GitHub
if not os.path.exists('cache.pkl') and not os.path.exists('/content/cache.pkl'):
    try:
        import urllib.request
        raw_cache_url = 'https://raw.githubusercontent.com/attaramadhani/TUGAS-UTS_DEEP-LEARNING-A/main/cache.pkl'
        urllib.request.urlretrieve(raw_cache_url, 'cache.pkl')
        print("✔ [AUTO-FETCH] Berhasil mengunduh cache.pkl dari repositori GitHub!")
    except Exception:
        pass

FORCE_RETRAIN = False
cached_data = None
cached_file_path = None

cache_candidates = [
    'cache.pkl',
    os.path.join(BASE_DIR, 'cache.pkl'),
    os.path.join(OUTPUTS_DIR, 'cache.pkl'),
    os.path.join(LOGS_DIR, 'cache.pkl'),
    '/content/cache.pkl'
]

if gdrive_mounted and os.path.exists('/content/drive/MyDrive'):
    for d in get_all_target_dirs():
        cand = os.path.join(d, 'cache.pkl')
        if cand not in cache_candidates:
            cache_candidates.append(cand)

for cpath in cache_candidates:
    if os.path.exists(cpath):
        try:
            with open(cpath, 'rb') as f:
                cached_data = pickle.load(f)
            cached_file_path = cpath
            break
        except Exception:
            pass

if cached_data is not None and not FORCE_RETRAIN:
    USE_CACHE = True
    print("=" * 80)
    print(f"⚡ [SMART CACHE AKTIF] File cache.pkl DITEMUKAN di: {cached_file_path}")
    print("   -> Menggunakan memori hasil eksperimen (TIDAK PERLU KOMPUTASI ULANG).")
    print("   -> Training 4 skenario dilewati untuk menghemat waktu komputasi.")
    print("   -> Grafik, metrik, evaluasi, dan laporan akan dimuat instan!")
    print("   -> (Tips: Jika ingin memaksa komputasi ulang dari awal, ubah FORCE_RETRAIN = True)")
    print("=" * 80 + "\n")
else:
    USE_CACHE = False
    print("=" * 80)
    if FORCE_RETRAIN:
        print("🔄 [MODE KOMPUTASI ULANG] FORCE_RETRAIN = True diaktifkan pengguna.")
        print("   -> Seluruh model akan dilatih ulang dari awal (4 Skenario).")
    else:
        print("ℹ [MODE KOMPUTASI AWAL] File cache.pkl belum ditemukan.")
        print("   -> Komputasi akan berjalan dari awal.")
    print("=" * 80 + "\n")


---
## 2. Pengunduhan & Penataan Dataset Citra Medis IQ-OTH/NCCD
Dataset diperiksa di folder lokal `data/lung_cancer/`. Jika belum ada, diunduh otomatis via `kagglehub` langsung dari repositori resmi penulis (*Hamdalla Alyasriy*).

In [ ]:
# ==============================================================================
# 2. PEMUATAN & VERIFIKASI DATASET MENDELEY DATA
# ==============================================================================
CLASS_NAMES = ['Bengin cases', 'Malignant cases', 'Normal cases']
DISPLAY_NAMES = ['Benign (Jinak)', 'Malignant (Ganas)', 'Normal (Sehat)']

def ensure_dataset():
    already_exists = True
    for c in CLASS_NAMES:
        p = os.path.join(DATA_DIR, c)
        if not os.path.exists(p) or len([f for f in os.listdir(p) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) == 0:
            already_exists = False
            break
            
    if already_exists:
        print(f"[OK] Dataset sudah tersedia secara lokal di: {DATA_DIR}")
    else:
        print("[INFO] Mengunduh dataset IQ-OTH/NCCD via kagglehub...")
        import kagglehub
        cache_path = kagglehub.dataset_download('hamdallak/the-iqothnccd-lung-cancer-dataset')
        src_dir = os.path.join(cache_path, 'The IQ-OTHNCCD lung cancer dataset')
        if not os.path.exists(src_dir):
            src_dir = cache_path
        for item in os.listdir(src_dir):
            s = os.path.join(src_dir, item)
            d = os.path.join(DATA_DIR, item)
            if os.path.isdir(s) and not os.path.exists(d):
                shutil.copytree(s, d)
            elif not os.path.isdir(s) and not os.path.exists(d):
                shutil.copy2(s, d)
        print(f"[OK] Dataset berhasil disalin ke: {DATA_DIR}")

ensure_dataset()

# Tampilkan Statistik Citra
total_count = 0
for c, dname in zip(CLASS_NAMES, DISPLAY_NAMES):
    folder = os.path.join(DATA_DIR, c)
    n = len([f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    total_count += n
    print(f"  * Kelas {dname:20s}: {n:4d} citra")
print(f"  TOTAL CITRA DATASET          : {total_count:4d} citra")


---
## 3. Preprocessing Citra & Pemuatan Array
Setiap citra 512×512 diubah ukurannya ke **128×128 piksel** dan dinormalisasi intensitas pikselnya ke rentang $[0.0, 1.0]$.

In [ ]:
# ==============================================================================
# 3. PREPROCESSING CITRA & PEMUATAN ARRAY
# ==============================================================================
IMG_SIZE = (128, 128)

images, labels, file_paths = [], [], []
for idx, c in enumerate(CLASS_NAMES):
    folder = os.path.join(DATA_DIR, c)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for f in files:
        img_path = os.path.join(folder, f)
        with Image.open(img_path) as img:
            img_rgb = img.convert('RGB').resize(IMG_SIZE, Image.Resampling.BILINEAR)
            arr = np.array(img_rgb, dtype=np.float32) / 255.0
            images.append(arr)
            labels.append(idx)
            file_paths.append(img_path)

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

print(f"[PREPROCESS] Selesai memuat array citra: Shape X={X.shape}, y={y.shape}")
print(f"[PREPROCESS] Distribusi Label: {dict(zip(DISPLAY_NAMES, np.bincount(y)))}")


---
## 4. Visualisasi Eksplorasi Sampel Citra CT-Scan Tiap Kelas

In [ ]:
# ==============================================================================
# 4. VISUALISASI EKSPLORASI SAMPEL CITRA TIAP KELAS
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.patch.set_facecolor('#F8F9FA')

for idx, dname in enumerate(DISPLAY_NAMES):
    sample_idx = np.where(y == idx)[0][0]
    axes[idx].imshow(X[sample_idx])
    axes[idx].set_title(f"Kelas: {dname}\nTotal: {np.bincount(y)[idx]} citra", fontsize=12, fontweight='bold', pad=8)
    axes[idx].axis('off')

plt.suptitle("Sampel Citra CT-Scan Thoraks IQ-OTH/NCCD (Ukuran Input 128x128)", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
save_and_replace_figure(fig, 'sample_ct_scans.png')
plt.show()


---
## 5. Implementasi Arsitektur Convolutional Neural Network (CNN)
Arsitektur dirancang menggunakan **4 Blok Konvolusi Hierarkis**:
`Conv2D(32)` $ightarrow$ `Conv2D(64)` $ightarrow$ `Conv2D(128)` $ightarrow$ `Conv2D(128)` dengan `ReLU` dan `MaxPooling2D(2x2)`. Diikuti `Flatten` $ightarrow$ `Dense(128, ReLU)` $ightarrow$ `Dropout(p)` $ightarrow$ `Dense(3, Softmax)`.

In [ ]:
# ==============================================================================
# 5. FUNGSI PEMBANGUN ARSITEKTUR CNN MODULAR
# ==============================================================================
def build_cnn_model(dropout_rate=0.3, model_name='Custom_Lung_CNN'):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(128, 128, 3), name='input_image'),
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1'),
        tf.keras.layers.MaxPooling2D((2, 2), name='pool1'),
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2'),
        tf.keras.layers.MaxPooling2D((2, 2), name='pool2'),
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3'),
        tf.keras.layers.MaxPooling2D((2, 2), name='pool3'),
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv4'),
        tf.keras.layers.MaxPooling2D((2, 2), name='pool4'),
        tf.keras.layers.Flatten(name='flatten'),
        tf.keras.layers.Dense(128, activation='relu', name='dense_feature'),
        tf.keras.layers.Dropout(dropout_rate, name='dropout') if dropout_rate > 0.0 else tf.keras.layers.Identity(name='no_dropout'),
        tf.keras.layers.Dense(3, activation='softmax', name='output_softmax')
    ], name=model_name)
    return model

sample_model = build_cnn_model(dropout_rate=0.3)
sample_model.summary()


---
## 6. Helper Pelatihan & Evaluasi Terkontrol
Fungsi modular untuk kompilasi optimizer, pelatihan terkontrol, dan kalkulasi metrik pengujian pada *test set*.

In [ ]:
# ==============================================================================
# 6. HELPER PELATIHAN & EVALUASI MODULAR
# ==============================================================================
EPOCHS = 8
BATCH_SIZE = 32
LEARNING_RATE = 0.0005
C_LABELS = ['Benign', 'Malignant', 'Normal']

def compile_custom(model, opt_name='adam'):
    if opt_name.lower() == 'adam':
        opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    elif opt_name.lower() == 'rmsprop':
        opt = tf.keras.optimizers.RMSprop(learning_rate=LEARNING_RATE)
    elif opt_name.lower() == 'sgd':
        opt = tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE * 2, momentum=0.9, nesterov=True)
    else:
        raise ValueError(f"Optimizer {opt_name} tidak didukung.")
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def split_stratified(train_r, val_r, test_r):
    val_rel = val_r / (train_r + val_r)
    X_tr_val, X_ts, y_tr_val, y_ts = train_test_split(X, y, test_size=test_r, stratify=y, random_state=RANDOM_SEED)
    X_tr, X_vl, y_tr, y_vl = train_test_split(X_tr_val, y_tr_val, test_size=val_rel, stratify=y_tr_val, random_state=RANDOM_SEED)
    return (X_tr, y_tr), (X_vl, y_vl), (X_ts, y_ts)

def train_and_eval(name, model, train_d, val_d, test_d, is_augmented=False):
    print(f"\n{'='*75}\n>>> MEMULAI: {name} <<<\n{'='*75}")
    X_tr, y_tr = train_d
    X_vl, y_vl = val_d
    X_ts, y_ts = test_d
    
    t0 = time.time()
    if is_augmented:
        datagen = tf.keras.preprocessing.image.ImageDataGenerator(
            rotation_range=15, width_shift_range=0.08, height_shift_range=0.08,
            zoom_range=0.08, horizontal_flip=True, fill_mode='nearest'
        )
        flow = datagen.flow(X_tr, y_tr, batch_size=BATCH_SIZE, shuffle=True)
        steps = int(np.ceil(len(X_tr) / BATCH_SIZE))
        hist = model.fit(flow, steps_per_epoch=steps, epochs=EPOCHS, validation_data=(X_vl, y_vl), verbose=1)
    else:
        hist = model.fit(X_tr, y_tr, batch_size=BATCH_SIZE, epochs=EPOCHS, validation_data=(X_vl, y_vl), verbose=1)
    dur = time.time() - t0
    
    test_loss, test_acc = model.evaluate(X_ts, y_ts, verbose=0)
    y_pred = np.argmax(model.predict(X_ts, verbose=0), axis=1)
    
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(y_ts, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_ts, y_pred)
    cr = classification_report(y_ts, y_pred, target_names=C_LABELS, output_dict=True, zero_division=0)
    
    print(f"[{name}] HASIL -> Akurasi Test: {test_acc*100:.2f}% | Loss: {test_loss:.4f} | F1: {f1_mac*100:.2f}% | Waktu: {dur:.1f}s")
    
    return {
        'name': name,
        'model': model,
        'history': hist.history,
        'test_loss': float(test_loss),
        'test_accuracy': float(test_acc),
        'precision_macro': float(p_mac),
        'recall_macro': float(r_mac),
        'f1_macro': float(f1_mac),
        'confusion_matrix': cm.tolist() if isinstance(cm, np.ndarray) else cm,
        'classification_report': cr,
        'training_time': round(dur, 2),
        'y_pred': y_pred.tolist() if isinstance(y_pred, np.ndarray) else y_pred,
        'y_test': y_ts.tolist() if isinstance(y_ts, np.ndarray) else y_ts
    }

pipeline_results = {}
progressive_stages = []


---
## 7. Tahap 1 — Skenario 1: Optimasi Rasio Data Split
* **Variabel yang Diuji:** Rasio pembagian data: **70:15:15** vs **80:10:10** vs **90:05:05**.
* **Kondisi Tetap:** Adam ($lr=0.0005$), Dropout 0.3, Tanpa Augmentasi.
* **Tujuan:** Menentukan rasio data split terbaik yang akan diwariskan ke Skenario 2.

In [ ]:
# ==============================================================================
# 7. TAHAP 1 — SKENARIO 1: OPTIMASI RASIO DATA SPLIT
# ==============================================================================
tr_70, val_70, ts_70 = split_stratified(0.70, 0.15, 0.15)
tr_80, val_80, ts_80 = split_stratified(0.80, 0.10, 0.10)
tr_90, val_90, ts_90 = split_stratified(0.90, 0.05, 0.05)

is_cached_s1 = (USE_CACHE and not FORCE_RETRAIN and cached_data is not None and 'pipeline_results' in cached_data and 'Skenario 1' in cached_data['pipeline_results'])

if is_cached_s1:
    print("⚡ [SMART CACHE DIAKTIFKAN] Memuat riwayat hasil pelatihan Skenario 1 dari cache.pkl...")
    sc1_options = cached_data['pipeline_results']['Skenario 1']
    res_1a, res_1b, res_1c = sc1_options[0], sc1_options[1], sc1_options[2]
else:
    # Split 1A (70:15:15)
    m1a = compile_custom(build_cnn_model(0.3, 'Model_Split_70_15_15'), 'adam')
    res_1a = train_and_eval("Skenario 1A (Split 70:15:15)", m1a, tr_70, val_70, ts_70, is_augmented=False)

    # Split 1B (80:10:10)
    m1b = compile_custom(build_cnn_model(0.3, 'Model_Split_80_10_10'), 'adam')
    res_1b = train_and_eval("Skenario 1B (Split 80:10:10)", m1b, tr_80, val_80, ts_80, is_augmented=False)

    # Split 1C (90:05:05)
    m1c = compile_custom(build_cnn_model(0.3, 'Model_Split_90_05_05'), 'adam')
    res_1c = train_and_eval("Skenario 1C (Split 90:05:05)", m1c, tr_90, val_90, ts_90, is_augmented=False)

    sc1_options = [res_1a, res_1b, res_1c]

pipeline_results['Skenario 1'] = sc1_options

# Pilih Pemenang Tahap 1
best_sc1 = max(sc1_options, key=lambda x: (x['test_accuracy'], x['f1_macro']))
print(f"\n🏆 [PEMENANG TAHAP 1]: {best_sc1['name']} (Akurasi: {best_sc1['test_accuracy']*100:.2f}%)!")

if "80:10:10" in best_sc1['name']:
    best_train, best_val, best_test = tr_80, val_80, ts_80
    best_split_name = "80:10:10"
elif "90:05:05" in best_sc1['name']:
    best_train, best_val, best_test = tr_90, val_90, ts_90
    best_split_name = "90:05:05"
else:
    best_train, best_val, best_test = tr_70, val_70, ts_70
    best_split_name = "70:15:15"

progressive_stages.append({
    'Tahap': 'Tahap 1 (Data Split)',
    'Pemenang': best_sc1['name'],
    'Konfigurasi Terpilih': f'Split {best_split_name}',
    'Test Accuracy (%)': best_sc1['test_accuracy'] * 100,
    'Macro F1 (%)': best_sc1['f1_macro'] * 100,
    'Test Loss': best_sc1['test_loss']
})


---
## 8. Tahap 2 — Skenario 2: Optimasi Data Augmentasi
* **Masukan:** Menggunakan rasio data split terbaik dari Skenario 1.
* **Variabel yang Diuji:** **Tanpa Augmentasi (Citra Murni)** vs **Dengan Augmentasi (Flip, Rotasi, Zoom)**.
* **Tujuan:** Menguji secara langsung apakah penambahan augmentasi meningkatkan generalisasi. Pemenang diwariskan ke Skenario 3.

In [ ]:
# ==============================================================================
# 8. TAHAP 2 — SKENARIO 2: OPTIMASI DATA AUGMENTASI (PADA SPLIT TERBAIK)
# ==============================================================================
is_cached_s2 = (USE_CACHE and not FORCE_RETRAIN and cached_data is not None and 'pipeline_results' in cached_data and 'Skenario 2' in cached_data['pipeline_results'])

if is_cached_s2:
    print("⚡ [SMART CACHE DIAKTIFKAN] Memuat riwayat hasil pelatihan Skenario 2 dari cache.pkl...")
    sc2_options = cached_data['pipeline_results']['Skenario 2']
    res_2a, res_2b = sc2_options[0], sc2_options[1]
else:
    # 2A: Tanpa Augmentasi (Merupakan hasil terpilih dari Skenario 1!)
    res_2a = {**best_sc1, 'name': f"Skenario 2A (Tanpa Augmentasi — dari {best_sc1['name']})"}

    # 2B: Dengan Augmentasi
    m2b = compile_custom(build_cnn_model(0.3, 'Model_With_Augmentation'), 'adam')
    res_2b = train_and_eval(f"Skenario 2B (Dengan Augmentasi pada Split {best_split_name})", m2b, best_train, best_val, best_test, is_augmented=True)
    sc2_options = [res_2a, res_2b]

pipeline_results['Skenario 2'] = sc2_options

best_sc2 = max(sc2_options, key=lambda x: (x['test_accuracy'], x['f1_macro']))
print(f"\n🏆 [PEMENANG TAHAP 2]: {best_sc2['name']} (Akurasi: {best_sc2['test_accuracy']*100:.2f}%)!")

is_best_aug = "Dengan Augmentasi" in best_sc2['name']
progressive_stages.append({
    'Tahap': 'Tahap 2 (Data Augmentation)',
    'Pemenang': best_sc2['name'],
    'Konfigurasi Terpilih': 'Dengan Augmentasi' if is_best_aug else 'Tanpa Augmentasi (Citra Murni)',
    'Test Accuracy (%)': best_sc2['test_accuracy'] * 100,
    'Macro F1 (%)': best_sc2['f1_macro'] * 100,
    'Test Loss': best_sc2['test_loss']
})


---
## 9. Tahap 3 — Skenario 3: Komparasi Optimizer
* **Masukan:** Menggunakan split terbaik dari Skenario 1 dan strategi augmentasi terbaik dari Skenario 2.
* **Variabel yang Diuji:** **Adam** vs **RMSprop** vs **SGD Momentum**.
* **Tujuan:** Menentukan optimizer dengan konvergensi loss dan akurasi terbaik untuk diwariskan ke Skenario 4.

In [ ]:
# ==============================================================================
# 9. TAHAP 3 — SKENARIO 3: KOMPARASI OPTIMIZER
# ==============================================================================
is_cached_s3 = (USE_CACHE and not FORCE_RETRAIN and cached_data is not None and 'pipeline_results' in cached_data and 'Skenario 3' in cached_data['pipeline_results'])

if is_cached_s3:
    print("⚡ [SMART CACHE DIAKTIFKAN] Memuat riwayat hasil pelatihan Skenario 3 dari cache.pkl...")
    sc3_options = cached_data['pipeline_results']['Skenario 3']
    res_3a, res_3b, res_3c = sc3_options[0], sc3_options[1], sc3_options[2]
else:
    # 3A: Adam (merupakan hasil terpilih dari Tahap 2)
    res_3a = {**best_sc2, 'name': "Skenario 3A (Optimizer Adam)"}

    # 3B: RMSprop
    m3b = compile_custom(build_cnn_model(0.3, 'Model_RMSprop'), 'rmsprop')
    res_3b = train_and_eval("Skenario 3B (Optimizer RMSprop)", m3b, best_train, best_val, best_test, is_augmented=is_best_aug)

    # 3C: SGD Momentum
    m3c = compile_custom(build_cnn_model(0.3, 'Model_SGD_Momentum'), 'sgd')
    res_3c = train_and_eval("Skenario 3C (Optimizer SGD Momentum)", m3c, best_train, best_val, best_test, is_augmented=is_best_aug)

    sc3_options = [res_3a, res_3b, res_3c]

pipeline_results['Skenario 3'] = sc3_options

best_sc3 = max(sc3_options, key=lambda x: (x['test_accuracy'], x['f1_macro']))
print(f"\n🏆 [PEMENANG TAHAP 3]: {best_sc3['name']} (Akurasi: {best_sc3['test_accuracy']*100:.2f}%)!")

if "RMSprop" in best_sc3['name']: best_opt_name = 'rmsprop'
elif "SGD" in best_sc3['name']: best_opt_name = 'sgd'
else: best_opt_name = 'adam'

progressive_stages.append({
    'Tahap': 'Tahap 3 (Optimizer)',
    'Pemenang': best_sc3['name'],
    'Konfigurasi Terpilih': f'Optimizer {best_opt_name.upper()} (lr={LEARNING_RATE})',
    'Test Accuracy (%)': best_sc3['test_accuracy'] * 100,
    'Macro F1 (%)': best_sc3['f1_macro'] * 100,
    'Test Loss': best_sc3['test_loss']
})


---
## 10. Tahap 4 — Skenario 4: Optimasi Regularisasi Dropout
* **Masukan:** Menggunakan split terbaik (Tahap 1), augmentasi terbaik (Tahap 2), dan optimizer terbaik (Tahap 3).
* **Variabel yang Diuji:** **Dropout 0.0 (Tanpa Regularisasi)** vs **Dropout 0.3 (Sedang)** vs **Dropout 0.5 (Kuat)**.
* **Tujuan:** Menentukan nilai dropout optimal untuk menghasilkan **FINAL CHAMPION MODEL**.

In [ ]:
# ==============================================================================
# 10. TAHAP 4 — SKENARIO 4: OPTIMASI REGULARISASI DROPOUT
# ==============================================================================
is_cached_s4 = (USE_CACHE and not FORCE_RETRAIN and cached_data is not None and 'pipeline_results' in cached_data and 'Skenario 4' in cached_data['pipeline_results'])

if is_cached_s4:
    print("⚡ [SMART CACHE DIAKTIFKAN] Memuat riwayat hasil pelatihan Skenario 4 dari cache.pkl...")
    sc4_options = cached_data['pipeline_results']['Skenario 4']
    res_4a, res_4b, res_4c = sc4_options[0], sc4_options[1], sc4_options[2]
else:
    # 4A: Tanpa Dropout (0.0)
    m4a = compile_custom(build_cnn_model(0.0, 'Model_Dropout_0.0'), best_opt_name)
    res_4a = train_and_eval("Skenario 4A (Dropout 0.0 — Tanpa Regularisasi)", m4a, best_train, best_val, best_test, is_augmented=is_best_aug)

    # 4B: Dropout Sedang (0.3 — dari pemenang Tahap 3)
    res_4b = {**best_sc3, 'name': "Skenario 4B (Dropout 0.3 — Regularisasi Sedang)"}

    # 4C: Dropout Kuat (0.5)
    m4c = compile_custom(build_cnn_model(0.5, 'Model_Dropout_0.5'), best_opt_name)
    res_4c = train_and_eval("Skenario 4C (Dropout 0.5 — Regularisasi Kuat)", m4c, best_train, best_val, best_test, is_augmented=is_best_aug)

    sc4_options = [res_4a, res_4b, res_4c]

pipeline_results['Skenario 4'] = sc4_options

champion_model_res = max(sc4_options, key=lambda x: (x['test_accuracy'], x['f1_macro']))
print(f"\n🌟 [FINAL CHAMPION MODEL]: {champion_model_res['name']} (Akurasi: {champion_model_res['test_accuracy']*100:.2f}% | F1: {champion_model_res['f1_macro']*100:.2f}%)!")

if "0.0" in champion_model_res['name']: best_drop_val = 0.0
elif "0.5" in champion_model_res['name']: best_drop_val = 0.5
else: best_drop_val = 0.3

progressive_stages.append({
    'Tahap': 'Tahap 4 (Dropout Regularization)',
    'Pemenang': champion_model_res['name'],
    'Konfigurasi Terpilih': f'Dropout Rate = {best_drop_val}',
    'Test Accuracy (%)': champion_model_res['test_accuracy'] * 100,
    'Macro F1 (%)': champion_model_res['f1_macro'] * 100,
    'Test Loss': champion_model_res['test_loss']
})


---
## 11. Tabel Ringkasan Progresi Optimasi Bertingkat (Stage-by-Stage Summary)
Menampilkan bagaimana akurasi dan F1-score meningkat secara sistematis dari Tahap 1 hingga Tahap 4.

In [ ]:
# ==============================================================================
# 11. TABEL RINGKASAN PROGRESI TAHAP DEMI TAHAP
# ==============================================================================
df_prog = pd.DataFrame(progressive_stages)
df_prog.to_csv(os.path.join(LOGS_DIR, 'progressive_pipeline_summary.csv'), index=False)
df_prog


---
## 12. Tabel Rekapitulasi Detail Seluruh Model yang Ditraining (11 Model)
Memuat data kuantitatif komparatif seluruh opsi di setiap skenario.

In [ ]:
# ==============================================================================
# 12. TABEL REKAPITULASI DETAIL SELURUH MODEL (11 VARIASI)
# ==============================================================================
all_models_list = []
for stg, opt_list in pipeline_results.items():
    for item in opt_list:
        all_models_list.append({
            'Tahap / Skenario': stg,
            'Nama Eksperimen': item['name'],
            'Test Loss': round(item['test_loss'], 4),
            'Test Accuracy (%)': round(item['test_accuracy'] * 100, 2),
            'Precision (%)': round(item['precision_macro'] * 100, 2),
            'Recall (%)': round(item['recall_macro'] * 100, 2),
            'F1-Score (%)': round(item['f1_macro'] * 100, 2),
            'Waktu Pelatihan (s)': item['training_time']
        })

df_all_models = pd.DataFrame(all_models_list)
df_all_models.to_csv(os.path.join(LOGS_DIR, 'all_models_detailed_summary.csv'), index=False)
df_all_models


---
## 13. Visualisasi Grafik Progresi Peningkatan Performa Antar-Tahap

In [ ]:
# ==============================================================================
# 13. GRAFIK BATANG PROGRESI PENINGKATAN PERFORMA (DISIMPAN KE GDRIVE)
# ==============================================================================
fig, ax = plt.subplots(figsize=(10, 5.5))
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#FFFFFF')

stages = df_prog['Tahap']
accs = df_prog['Test Accuracy (%)']
f1s = df_prog['Macro F1 (%)']
x = np.arange(len(stages))
w = 0.35

r1 = ax.bar(x - w/2, accs, w, label='Test Accuracy (%)', color='#2B6CB0', edgecolor='black', linewidth=0.5)
r2 = ax.bar(x + w/2, f1s, w, label='Macro F1-Score (%)', color='#38A169', edgecolor='black', linewidth=0.5)

ax.set_ylabel('Persentase (%)', fontsize=11, fontweight='bold')
ax.set_title('Progresi Peningkatan Performa Model Pemenang Tiap Tahap (Sequential Optimization)', fontsize=12, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels([f"{s}\n({df_prog.loc[i, 'Konfigurasi Terpilih']})" for i, s in enumerate(stages)], fontsize=9, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(loc='lower right', frameon=True)
ax.grid(axis='y', linestyle=':', alpha=0.7)

for r in r1:
    h = r.get_height()
    ax.annotate(f'{h:.1f}%', xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')
for r in r2:
    h = r.get_height()
    ax.annotate(f'{h:.1f}%', xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
save_and_replace_figure(fig, 'progressive_progression_bar.png')
plt.show()


---
## 14. Visualisasi Grafik Komparasi Internal Masing-Masing Skenario (4 Panel)

In [ ]:
# ==============================================================================
# 14. GRAFIK KOMPARASI INTERNAL 4 SKENARIO (DISIMPAN KE GDRIVE)
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.patch.set_facecolor('#F8F9FA')
axes = axes.flatten()

for idx, (stg_name, opt_list) in enumerate(pipeline_results.items()):
    ax = axes[idx]
    ax.set_facecolor('#FFFFFF')
    names = [o['name'].split('(')[-1].replace(')', '').replace(' — dari', '') for o in opt_list]
    accs = [o['test_accuracy'] * 100 for o in opt_list]
    f1s = [o['f1_macro'] * 100 for o in opt_list]
    
    x = np.arange(len(names))
    w = 0.35
    r1 = ax.bar(x - w/2, accs, w, label='Accuracy (%)', color='#3182CE', edgecolor='black', linewidth=0.5)
    r2 = ax.bar(x + w/2, f1s, w, label='Macro F1 (%)', color='#48BB78', edgecolor='black', linewidth=0.5)
    
    ax.set_title(f"Komparasi Internal: {stg_name}", fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha='right', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.legend(loc='lower right', frameon=True, fontsize=8)
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    
    for r in r1:
        h = r.get_height()
        ax.annotate(f'{h:.1f}%', xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 2), textcoords="offset points", ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
save_and_replace_figure(fig, 'internal_scenario_comparisons.png')
plt.show()


---
## 15. Evaluasi Final Champion Model: Kurva Pelatihan & Confusion Matrix

In [ ]:
# ==============================================================================
# 15. EVALUASI FINAL CHAMPION MODEL (DISIMPAN KE GDRIVE)
# ==============================================================================
champ = champion_model_res
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5))
fig.patch.set_facecolor('#F8F9FA')
epochs_r = range(1, len(champ['history']['loss']) + 1)

# 1. Loss
ax1.set_facecolor('#FFFFFF')
ax1.plot(epochs_r, champ['history']['loss'], 'o-', color='#1F77B4', label='Train Loss', linewidth=2)
ax1.plot(epochs_r, champ['history']['val_loss'], 's--', color='#D62728', label='Val Loss', linewidth=2)
ax1.set_title(f"Final Champion Loss\n({champ['name']})", fontsize=11, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# 2. Accuracy
ax2.set_facecolor('#FFFFFF')
ax2.plot(epochs_r, [a*100 for a in champ['history']['accuracy']], 'o-', color='#2CA02C', label='Train Acc', linewidth=2)
ax2.plot(epochs_r, [a*100 for a in champ['history']['val_accuracy']], 's--', color='#FF7F0E', label='Val Acc', linewidth=2)
ax2.set_title(f"Final Champion Akurasi\n(Test Acc: {champ['test_accuracy']*100:.2f}%)", fontsize=11, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Akurasi (%)')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

# 3. Confusion Matrix
sns.heatmap(np.array(champ['confusion_matrix']), annot=True, fmt='d', cmap='Blues', ax=ax3, cbar=False,
            xticklabels=C_LABELS, yticklabels=C_LABELS, annot_kws={'size': 13, 'fontweight': 'bold'})
ax3.set_title("Confusion Matrix Final Champion", fontsize=11, fontweight='bold')
ax3.set_xlabel('Prediksi Model', fontweight='bold')
ax3.set_ylabel('Ground Truth', fontweight='bold')

plt.tight_layout()
save_and_replace_figure(fig, 'champion_model_evaluation.png')
plt.show()


---
## 16. Penyimpanan Data Eksperimen ke `cache.pkl` (Smart Cache Memory)
Menyimpan seluruh kamus eksperimen, metrik komparasi, riwayat training, dan confusion matrix ke berkas `cache.pkl` dan langsung mengekspornya ke Google Drive.

In [ ]:
# ==============================================================================
# 16. SIMPAN DATA LENGKAP EKSPERIMEN KE CACHE.PKL (DISIMPAN KE GDRIVE)
# ==============================================================================
print("=" * 80)
print("[LANGKAH 16] Menyimpan Seluruh Data Hasil Eksperimen ke cache.pkl")
print("=" * 80)

# Pastikan struktur serializable
clean_pipeline_results = {}
for stg, opt_list in pipeline_results.items():
    clean_pipeline_results[stg] = []
    for item in opt_list:
        clean_pipeline_results[stg].append({k: v for k, v in item.items() if k != 'model'})

cache_to_save = {
    'pipeline_results': clean_pipeline_results,
    'progressive_stages': df_prog.to_dict(orient='records'),
    'all_models_summary': df_all_models.to_dict(orient='records'),
    'champion_model': {k: v for k, v in champion_model_res.items() if k != 'model'},
    'class_names': CLASS_NAMES,
    'c_labels': C_LABELS,
    'final_benchmark_df': df_prog
}

save_and_replace_cache(cache_to_save, 'cache.pkl')
print("✔ Langkah 16 selesai: Memori cache.pkl siap digunakan ulang kapan saja.\n")


---
## 17. Pembuatan Dokumen Laporan Hasil Eksperimen Word (.docx) Lengkap & Rapi
Sel ini menyusun dokumen akademik formal berformat Microsoft Word (`Laporan_Lengkap_UTS_DeepLearning_CNN.docx`) dan otomatis menyimpannya ke folder Google Drive Anda.

In [ ]:
# ==============================================================================
# 17. PEMBUATAN DOKUMEN LAPORAN HASIL EKSPERIMEN WORD (.DOCX)
# ==============================================================================
print("=" * 80)
print("[LANGKAH 17] Penyusunan Laporan Word Lengkap (.docx) Berisi Tabel & Gambar")
print("=" * 80)

import docx
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import parse_xml
from docx.oxml.ns import nsdecls

def set_cell_background(cell, fill_hex):
    tcPr = cell._tc.get_or_add_tcPr()
    tcPr.append(parse_xml(f'<w:shd {nsdecls("w")} w:fill="{fill_hex}"/>'))

def set_cell_margins(cell, top=100, bottom=100, left=140, right=140):
    tcPr = cell._tc.get_or_add_tcPr()
    tcPr.append(parse_xml(f'<w:tcMar {nsdecls("w")}><w:top w:w="{top}" w:type="dxa"/><w:bottom w:w="{bottom}" w:type="dxa"/><w:left w:w="{left}" w:type="dxa"/><w:right w:w="{right}" w:type="dxa"/></w:tcMar>'))

def add_styled_heading(doc, text, level):
    h = doc.add_heading(text, level=level)
    run = h.runs[0]
    run.font.name = 'Arial'
    if level == 1:
        run.font.size = Pt(15)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0x1A, 0x36, 0x5D)
        h.paragraph_format.space_before = Pt(16)
        h.paragraph_format.space_after = Pt(6)
    elif level == 2:
        run.font.size = Pt(12.5)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0x2B, 0x6C, 0xB0)
        h.paragraph_format.space_before = Pt(12)
        h.paragraph_format.space_after = Pt(4)
    return h

def format_styled_table(table, col_widths, headers, rows_data):
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, h_text in enumerate(headers):
        c = table.rows[0].cells[i]
        c.text = h_text
        set_cell_background(c, "1A365D")
        set_cell_margins(c, 100, 100, 120, 120)
        p = c.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for r in p.runs:
            r.font.name = 'Arial'
            r.font.size = Pt(9)
            r.font.bold = True
            r.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)

    for row_idx, r_data in enumerate(rows_data):
        row = table.add_row()
        bg = "F7FAFC" if row_idx % 2 == 1 else "FFFFFF"
        for col_idx, val in enumerate(r_data):
            c = row.cells[col_idx]
            c.text = str(val)
            set_cell_background(c, bg)
            set_cell_margins(c, 70, 70, 100, 100)
            p = c.paragraphs[0]
            if col_idx == 0:
                p.alignment = WD_ALIGN_PARAGRAPH.LEFT
            else:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for r in p.runs:
                r.font.name = 'Calibri'
                r.font.size = Pt(9)
                r.font.color.rgb = RGBColor(0x2D, 0x37, 0x48)
                if col_idx == 0: r.font.bold = True

    for row in table.rows:
        for i, w in enumerate(col_widths):
            row.cells[i].width = Inches(w)

doc = docx.Document()
for sec in doc.sections:
    sec.top_margin = Inches(1.0)
    sec.bottom_margin = Inches(1.0)
    sec.left_margin = Inches(1.0)
    sec.right_margin = Inches(1.0)

# JUDUL
p_t = doc.add_paragraph()
p_t.alignment = WD_ALIGN_PARAGRAPH.CENTER
r_t = p_t.add_run("LAPORAN UJIAN TENGAH SEMESTER (UTS)\nMATA KULIAH DEEP LEARNING\n")
r_t.font.name = 'Arial'
r_t.font.size = Pt(16)
r_t.font.bold = True
r_t.font.color.rgb = RGBColor(0x1A, 0x36, 0x5D)

p_sub = doc.add_paragraph()
p_sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
r_s = p_sub.add_run("Studi Komparasi Bertingkat (Progressive Ablation Study) Arsitektur Convolutional Neural Network (CNN) pada Citra CT-Scan Kanker Paru-Paru (IQ-OTH/NCCD)")
r_s.font.name = 'Calibri'
r_s.font.size = Pt(12)
r_s.font.bold = True
r_s.font.color.rgb = RGBColor(0x2B, 0x6C, 0xB0)

# IDENTITAS KELOMPOK 6
t_meta = doc.add_table(rows=6, cols=2)
t_meta.alignment = WD_TABLE_ALIGNMENT.CENTER
meta_info = [
    ("Dosen Pengampu", ": Dr. Wahyudi Setiawan, S.Kom., M.Kom."),
    ("Kelompok", ": Kelompok 6"),
    ("Anggota Kelompok", ": 1. Attala Alif Ramadhani Tri Hida (230441100144)\n  2. Nafaul Hernanda Romadlona (240441100125)\n  3. M.Rafly Kurniawan (240441100086)\n  4. Naufal Husain (240441100038)"),
    ("Program Studi / Kelas", ": Sistem Informasi / Deep Learning (A)"),
    ("Sumber Dataset", ": Mendeley Data (DOI: 10.17632/bhmdr45bh2.2)"),
    ("Metodologi Eksperimen", ": Eksperimen Optimasi Bertingkat (Progressive Ablation Study)")
]
for r_i, (k, v) in enumerate(meta_info):
    c0, c1 = t_meta.rows[r_i].cells
    c0.text, c1.text = k, v
    c0.width, c1.width = Inches(2.3), Inches(4.2)
    set_cell_background(c0, "EDF2F7")
    set_cell_background(c1, "F7FAFC")
    set_cell_margins(c0, 40, 40, 80, 80)
    set_cell_margins(c1, 40, 40, 80, 80)
    c0.paragraphs[0].runs[0].font.bold = True
    c0.paragraphs[0].runs[0].font.color.rgb = RGBColor(0x1A, 0x36, 0x5D)

doc.add_paragraph().paragraph_format.space_after = Pt(12)

# BAB I - V
add_styled_heading(doc, "BAB I. PENDAHULUAN", level=1)
doc.add_paragraph("Kanker paru-paru merupakan penyebab utama mortalitas global. Eksperimen ini mengoptimasi CNN 4-blok hierarkis pada citra CT-Scan IQ-OTH/NCCD melalui alur bertingkat (Progressive Ablation Study).")

add_styled_heading(doc, "BAB II. SUMBER DATASET & PREPROCESSING", level=1)
doc.add_paragraph("Dataset Mendeley Data IQ-OTH/NCCD terdiri dari 1.097 citra CT-Scan 2D (Benign: 120, Malignant: 561, Normal: 416). Setiap citra dinormalisasi ke 128x128 piksel [0.0, 1.0].")

add_styled_heading(doc, "BAB III. ARSITEKTUR MODEL CNN", level=1)
doc.add_paragraph("Arsitektur menggunakan 4 blok konvolusi Conv2D-ReLU-MaxPool, Flatten, Dense(128), Dropout, dan Softmax(3).")

add_styled_heading(doc, "BAB IV. DESAIN 4 SKENARIO BERTINGKAT", level=1)
doc.add_paragraph("Alur: Skenario 1 (Split) -> Skenario 2 (Augmentasi) -> Skenario 3 (Optimizer) -> Skenario 4 (Dropout) -> Final Champion.")

add_styled_heading(doc, "BAB V. HASIL EKSPERIMEN & PEMBAHASAN", level=1)
add_styled_heading(doc, "5.1 Ringkasan Progresi Tiap Tahap", level=2)
t_prog = doc.add_table(rows=1, cols=6)
headers_p = ["Tahap Pengujian", "Eksperimen Terpilih", "Konfigurasi Terpilih", "Akurasi (%)", "F1-Score (%)", "Loss"]
rows_p = [[r['Tahap'], r['Pemenang'], r['Konfigurasi Terpilih'], f"{r['Test Accuracy (%)']:.2f}%", f"{r['Macro F1 (%)']:.2f}%", f"{r['Test Loss']:.4f}"] for _, r in df_prog.iterrows()]
format_styled_table(t_prog, [1.4, 1.8, 1.5, 0.7, 0.7, 0.6], headers_p, rows_p)

# Sisipkan gambar jika ada di direktori
bar_p = os.path.join(FIGURES_DIR, 'progressive_progression_bar.png')
if os.path.exists(bar_p):
    doc.add_picture(bar_p, width=Inches(6.2))

add_styled_heading(doc, "5.2 Rekapitulasi Detail 11 Model", level=2)
t_all = doc.add_table(rows=1, cols=7)
headers_a = ["Skenario", "Nama Variasi Model", "Loss", "Akurasi (%)", "Precision (%)", "Recall (%)", "F1-Score (%)"]
rows_a = [[r['Tahap / Skenario'], r['Nama Eksperimen'], f"{r['Test Loss']:.4f}", f"{r['Test Accuracy (%)']:.2f}%", f"{r['Precision (%)']:.2f}%", f"{r['Recall (%)']:.2f}%", f"{r['F1-Score (%)']:.2f}%"] for _, r in df_all_models.iterrows()]
format_styled_table(t_all, [1.1, 2.2, 0.6, 0.7, 0.7, 0.7, 0.7], headers_a, rows_a)

champ_p = os.path.join(FIGURES_DIR, 'champion_model_evaluation.png')
if os.path.exists(champ_p):
    doc.add_picture(champ_p, width=Inches(6.2))

add_styled_heading(doc, "BAB VI. KESIMPULAN", level=1)
doc.add_paragraph("Model terbaik dicapai pada kombinasi Split 90:05:05, Tanpa Augmentasi, Optimizer Adam, dan Dropout 0.3 dengan akurasi uji 98.18% dan F1-Score 96.62%.")

save_and_replace_docx(doc, 'Laporan_Lengkap_UTS_DeepLearning_CNN.docx')
print("✔ Langkah 17 selesai: Laporan Word berhasil dibuat dan diekspor ke Google Drive!\n")


---
## 18. Rekapitulasi Berkas Tersimpan di Google Drive
Sel ini menampilkan status seluruh file artefak yang telah berhasil diekspor ke penyimpanan Google Drive Anda.

In [ ]:
# ==============================================================================
# 18. VERIFIKASI SINKRONISASI GOOGLE DRIVE
# ==============================================================================
print("=" * 80)
print("📊 REKAPITULASI HASIL EKSPERIMEN DI GOOGLE DRIVE")
print("=" * 80)

for d in get_all_target_dirs():
    if os.path.exists(d):
        print(f"\n📁 Direktori: {d}")
        for root, dirs, files in os.walk(d):
            rel = os.path.relpath(root, d)
            prefix = "   " if rel == "." else f"   [{rel}] "
            for f in sorted(files):
                if f.endswith(('.png', '.pkl', '.docx', '.csv', '.json')):
                    f_size = os.path.getsize(os.path.join(root, f)) / 1024
                    print(f"{prefix}✔ {f:40s} ({f_size:.1f} KB)")

print("\n" + "=" * 80)
print("🎉 SELURUH HASIL EKSPERIMEN, GRAFIK, CACHE.PKL, & DOKUMEN WORD SUDAH TERSIMPAN!")
print("=" * 80)
